# Vignette 2: Fine-tune a LoRA on your own dermatology data

The paper fine-tuned Stable Diffusion with DreamBooth, one model per disease. In 2026 the
standard approach is a LoRA adapter trained with the upstream diffusers scripts, one adapter for
all labels with the disease name in the caption. `synderm.fine_tune.train_lora` exports your
dataset in the format those scripts read and launches them with `accelerate`.

This notebook trains a text-to-image LoRA for Qwen-Image-2.1 on the six-label sample, generates
images with it, and shows them next to (a) real images and (b) the 2024 Stable Diffusion 2.1
DreamBooth generations from the public `synthetic-derm-1M` dataset for the same labels.

In [ ]:
SMALL = True
BACKEND = "qwen-image-2.1"
RESOLUTION = 512 if SMALL else 768
MAX_TRAIN_STEPS = 200 if SMALL else 1000
RANK = 16
NUM_PER_LABEL = 4
SEED = 0

In [ ]:
import os, json, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
WORK = Path(os.environ.get("SYNDERM_WORK_DIR", "/workspace/synderm"))
WORK.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

def show_grid(rows, row_titles=None, col_titles=None, size=2.6, suptitle=None):
    """rows: list of lists of PIL images (one list per row)."""
    n_rows, n_cols = len(rows), max(len(r) for r in rows)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, size * n_rows), squeeze=False)
    for r, row in enumerate(rows):
        for c in range(n_cols):
            ax = axes[r][c]; ax.axis("off")
            if c < len(row) and row[c] is not None:
                ax.imshow(row[c])
            if r == 0 and col_titles and c < len(col_titles):
                ax.set_title(col_titles[c], fontsize=10)
        if row_titles:
            axes[r][0].text(-0.08, 0.5, row_titles[r], transform=axes[r][0].transAxes, rotation=90,
                            va="center", ha="right", fontsize=10)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout(); plt.show()

## 1. The training data

Any dataset whose items have `id`, `image` and `label` works. Here it is the Atlas Dermatologico
sample (64 images per label).

In [ ]:
from synderm.data import load_sample
train = load_sample(split="train")
labels = train.labels
print(len(train), "images,", len(labels), "labels:", labels)

## 2. Train the adapter

`train_lora` writes `train-data/` (a parquet with image and caption columns), `train.log`,
`train_config.json` (the exact command and git commit) and finally
`pytorch_lora_weights.safetensors` into the output directory. On one RTX 5090 the small
setting takes roughly 10 to 15 minutes; it raises `TrainingError` with the log tail on failure.

In [ ]:
from synderm.fine_tune import train_lora
lora_dir = WORK / "lora" / f"{BACKEND}-t2i-r{RANK}-{RESOLUTION}px-{MAX_TRAIN_STEPS}steps"
weights = lora_dir / "pytorch_lora_weights.safetensors"
if not weights.exists():
    t = time.time()
    weights = train_lora(BACKEND, train, lora_dir, mode="text-to-image", resolution=RESOLUTION,
                         max_train_steps=MAX_TRAIN_STEPS, rank=RANK, seed=SEED)
    print(f"trained in {(time.time() - t) / 60:.1f} min")
print(weights)
print(json.dumps(json.load(open(lora_dir / "train_config.json")), indent=1)[:800])

## 3. Generate with the adapter

`get_backend(..., lora_path=...)` applies the adapter. `generate_synthetic_dataset` writes
`<out>/<label>/<id>.png` plus a JSON with the prompt, seed, backend and adapter, in the layout
the rest of synderm (and the Hugging Face shard builder) understands.

In [ ]:
from synderm.backends import get_backend
from synderm.generation import generate_synthetic_dataset
gen = get_backend(BACKEND, lora_path=weights)
out_dir = WORK / "generations" / f"{BACKEND}-lora-t2i"
syn = generate_synthetic_dataset(gen, out_dir, method="text-to-image", labels=labels, num_per_label=NUM_PER_LABEL,
                                 height=RESOLUTION, width=RESOLUTION, seed=SEED, batch_size=NUM_PER_LABEL)
gen.unload()
print(len(syn), "synthetic images in", out_dir)

## 4. Old vs new

The 2024 public dataset was generated with Stable Diffusion 2.1 fine-tuned with DreamBooth on
the full Fitzpatrick 17k class at 512 px; the images above come from a LoRA on 64 images per
label. The comparison is qualitative and not controlled, but it shows what three years of
generator progress look like at the point where you have just trained your own.
`load_2024_shards` downloads one shard per label from `tbuckley/synthetic-derm-1M` (1 to 2.5 GB
each) and unpacks a bounded number of samples.

In [ ]:
from synderm.data import load_2024_shards
old = load_2024_shards(labels, method="finetune-text-to-image", submethod="text-to-image", max_per_label=NUM_PER_LABEL)
def first_per_label(ds, labels, n):
    out = {l: [] for l in labels}
    for i in range(len(ds)):
        lab = ds.items[i][1]["label"] if hasattr(ds, "items") else ds[i]["label"]
        if lab in out and len(out[lab]) < n:
            out[lab].append(ds[i]["image"])
    return out
real_ex, old_ex, new_ex = first_per_label(train, labels, 1), first_per_label(old, labels, 2), first_per_label(syn, labels, 2)
rows, titles = [], []
for l in labels:
    rows.append([real_ex[l][0]] + old_ex[l] + new_ex[l])
    titles.append(l)
show_grid(rows, row_titles=titles, col_titles=["real", "SD 2.1 DreamBooth (2024)", "", f"{BACKEND} LoRA (now)", ""], size=2.2)

## Next

- Vignette 3 conditions generation on a real image instead of text alone.
- Vignette 4 and 5 train classifiers with these images and measure whether they help.
- The same call with `mode="image-to-image"` trains an editing adapter (Qwen and FLUX only).